In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass
from tqdm import trange
from dataclasses import dataclass
from sklearn.model_selection import train_test_split
from utils import stratified_train_test_split

# Implement a Decision Tree (alternative version)

In [ ]:
from typing import Optional, List
from utils import print_tree


@dataclass
class Node:
    feature: Optional[int] = None  # Index of the feature used for split; None if leaf
    threshold: Optional[float] = None  # Threshold value for split; None if leaf
    prediction: Optional[float] = (
        None  # Predicted class (majority class); defined for leaves
    )
    left_node: Optional["Node"] = None  # Left child Node object; None if leaf
    right_node: Optional["Node"] = None  # Right child Node object; None if leaf
    is_leaf: bool = False

    # NOTE: This is optional, if you don't define it, fit and predict still work; it's only for visualization
    def __repr__(self, level=0, prefix=""):
        return print_tree(self, level=level, prefix=prefix)
        

def gini_impurity(y):
    class_counts = np.unique(y, return_counts=True)[1]
    probs = class_counts / len(y)
    return 1 - np.sum(probs**2)


def split_gini(y_left, y_right):
    n_left = len(y_left)
    n_right = len(y_right)
    n_total = n_left + n_right
    gini_left = gini_impurity(y_left)
    gini_right = gini_impurity(y_right)
    return (n_left / n_total) * gini_left + (n_right / n_total) * gini_right


def find_best_split_feature_and_threshold(X, y, cat_features_idx):
    """Find the best feature and threshold to split the data based on Gini impurity."""
    n_samples, n_features = X.shape

    best_gini = np.inf  # we want to minimize the Gini Impurity
    best_feature = None
    best_threshold = None

    # try one feature at a time
    for feature_idx in range(n_features):
        # candidates thresholds are all the possible unique values of the feature in the training set
        candidate_thresholds = np.unique(X[:, feature_idx])

        # check if the feature is categorical: this will determine how we split the data, given the threshold
        # if categorical, we will split data into 2 groups: the one with samples for which their feature value
        # is equal to the threshold, and the one for which it is different
        # if continuous, we will split data into 2 groups: the one with samples for which their feature value
        # is less than or equal to the threshold, and the one for which it is greater than the threshold
        is_cat = feature_idx in cat_features_idx

        # try all possible thresholds
        for threshold in candidate_thresholds:
            # potential split of the dataset
            if is_cat:
                binary_mask_left_child = X[:, feature_idx] == threshold
                y_left_child = y[binary_mask_left_child]

                binary_mask_right_child = X[:, feature_idx] != threshold
                y_right_child = y[binary_mask_right_child]
            else:
                binary_mask_left_child = X[:, feature_idx] <= threshold
                y_left_child = y[binary_mask_left_child]

                binary_mask_right_child = X[:, feature_idx] > threshold
                y_right_child = y[binary_mask_right_child]

            # compute potential Gini impurity of the split
            current_gini = split_gini(y_left_child, y_right_child)
            if current_gini < best_gini:
                best_gini = current_gini
                best_feature = feature_idx
                best_threshold = threshold
            # else we evaluate the next threshold and eventually the next feature

    return best_feature, best_threshold


def fit_decision_tree(
    X_train, y_train, max_depth, min_impurity_improvement, cat_features_idx
):
    """Fit a decision tree classifier to the training data.
    We use an iterative approach to build the tree.
    The tree is represented by the Node class, which defines the root.
    Each Node has two attributes: left and right, representing its child nodes.
    These child nodes are also instances of the Node class, recursively forming the tree structure.

    Parameters
    ----------
    X_train : np.ndarray
        The training data, shape (n_samples, n_features)
    y_train : np.ndarray
        The target values, shape (n_samples,)
    max_depth : int
        The maximum depth of the tree
    min_impurity_improvement : float
        The minimum impurity improvement required to split a node
    cat_features_idx : list of int
        The indices of the categorical features in X_train. If
    """
    root = Node(
        feature=None,
        threshold=None,
        prediction=None,
        left_node=None,
        right_node=None,
        is_leaf=False,
    )

    queue = [(0, X_train, y_train, root)]  # (depth, X_subset, y_subset, node object)

    while len(queue) > 0:
        depth, X_subset, y_subset, node = queue.pop(0)

        # prediction of the node is the mode
        classes, counts = np.unique(y_subset, return_counts=True)
        prediction = classes[np.argmax(counts)]

        # check if the node contains <=1 samples. If so, we set it as a leaf
        is_leaf = depth >= max_depth or np.unique(y_subset).shape[0] <= 1
        if is_leaf:
            node.is_leaf = True
            node.prediction = prediction
            continue  # Skip to the next node in the queue

        # find the best feature and threshold to split the data
        split_feature, split_threshold = find_best_split_feature_and_threshold(
            X_subset, y_subset, cat_features_idx
        )

        # divide the data into 2 children using split_feature and split_threshold.
        # the split depends on whether the feature is categorical or continuous
        is_cat = split_feature in cat_features_idx

        if is_cat:
            mask_left_child = X_subset[:, split_feature] == split_threshold
            X_left_child = X_subset[mask_left_child]
            y_left_child = y_subset[mask_left_child]

            mask_right_child = X_subset[:, split_feature] != split_threshold
            X_right_child = X_subset[mask_right_child]
            y_right_child = y_subset[mask_right_child]

        else:  # if continous
            mask_left_child = X_subset[:, split_feature] <= split_threshold
            X_left_child = X_subset[mask_left_child]
            y_left_child = y_subset[mask_left_child]

            mask_right_child = X_subset[:, split_feature] > split_threshold
            X_right_child = X_subset[mask_right_child]
            y_right_child = y_subset[mask_right_child]

        # check if the improvement is too small: if so, we set the node as a leaf
        impurity_improvement = gini_impurity(y_subset) - split_gini(
            y_left_child, y_right_child
        )
        is_leaf_2 = impurity_improvement < min_impurity_improvement
        if is_leaf_2:
            node.is_leaf = True
            node.prediction = prediction
            continue

        # else we can go on with splitting.
        # update the node
        node.feature = split_feature
        node.threshold = split_threshold
        node.prediction = prediction
        node.left_node = Node(
            feature=None,
            threshold=None,
            prediction=None,
            left_node=None,
            right_node=None,
        )
        node.right_node = Node(
            feature=None,
            threshold=None,
            prediction=None,
            left_node=None,
            right_node=None,
        )
        node.is_leaf = False

        # add left and right children to the queue
        if len(X_left_child) > 0 and len(X_right_child) > 0:
            queue.append((depth + 1, X_left_child, y_left_child, node.left_node))
            queue.append((depth + 1, X_right_child, y_right_child, node.right_node))

    return root

In [ ]:
data = pd.read_csv('ML_resources/iris.csv')
display(data.head())

feature_names = data.columns[:-1]
target_col = 'species'

X = data.drop(target_col, axis=1).to_numpy()
y = data[target_col].to_numpy()

print("X.shape_", X.shape, "y.shape_", y.shape)

# convert the target

# the tree is now represented by the root node: it is a Node object that contains all the other
# nodes of the tree (in a nested way).
tree = fit_decision_tree(X, y, max_depth=3, min_impurity_improvement=0.0, cat_features_idx=[])
tree

In [ ]:
def predict(root_node, X, cat_features_idx): 
    """Predict the class of the input data using the decision tree."""
    predictions = []
    for sample_x in X: 
        node = root_node
        while not node.is_leaf: 
            if node.feature in cat_features_idx:
                if sample_x[node.feature] == node.threshold:
                    node = node.left_node
                else:
                    node = node.right_node
            else: # continous feature
                if sample_x[node.feature] <= node.threshold:
                    node = node.left_node
                else:
                    node = node.right_node
        predictions.append(node.prediction)
    return np.array(predictions)

In [ ]:
def accuracy(y_true, y_pred):
    """
    Args:
    y_true: array of true labels. Shape (n_samples, ) or (n_samples, 1)
    y_pred: array of predicted labels. Shape (n_samples, ) or (n_samples, 1)
    """
    total_predictions = len(y_true)
    is_correct = y_true == y_pred
    n_correct_pred = np.sum(is_correct)
    accuracy = n_correct_pred / total_predictions
    return accuracy

In [ ]:
y_pred = predict(tree, X, cat_features_idx=[])
print("Training accuracy:", accuracy(y, y_pred))

# Cross-Validation

We use cross-validation when we want to have an estimate of the performance of our model, and we do not want the testing to be affected by the specific training-test split we chose.

In **K-fold cross-validation**, the dataset is divided into *K* equal-sized subsets, or "folds". The process works as follows:

1. For each iteration, one fold is held out as the test set.
2. The model is trained on the remaining *K - 1* folds.
3. The trained model is evaluated on the test fold.
4. This process is repeated *K* times, with each fold used exactly once for testing.

The final performance metric is the average of the metrics obtained from all *K* iterations. After evaluation, a final model is typically trained on the full dataset for use in real-world predictions.

Cross-validation can be performed with almost any model we have studied. 

The only caveat is to watch out for high computation times when using more complex models, such as XGBoost or neural networks.

## Example with scikit-learn

In [ ]:
from sklearn.model_selection import KFold

# Set up cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
accuracies = []

for train_index, test_index in kf.split(X): # kf.split(X) returns the indices of the training and test sets
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]
    
    # Train the tree on the training fold
    tree = fit_decision_tree(X_train, y_train, max_depth=3, min_impurity_improvement=0.0, cat_features_idx=[])
    
    # Predict on the validation fold
    y_pred = predict(tree, X_test, cat_features_idx=[])
    
    # Evaluate accuracy
    acc = accuracy(y_test, y_pred)
    accuracies.append(acc)
    print(f"Fold accuracy: {acc:.4f}")

# Report average performance
print(f"\nAverage cross-validation accuracy: {np.mean(accuracies):.4f}")


## Example without scikit-learn

In [ ]:
import numpy as np

# General fold generator function
def get_folds(X, y, n_splits=5, seed=42):
    np.random.seed(seed)
    # shuffle the indices
    indices = np.random.permutation(len(X))

    # create the folds
    total_samples = len(X)
    # size of all folds if the dataset is divisible by n_splits
    base_fold_size = total_samples // n_splits
    # if total_samples is not divisible by n_splits, we need to handle the remainder
    remainder = total_samples % n_splits

    fold_sizes = []
    for i in range(n_splits):
        # distribute the remainder across the first few folds
        # e.g. if n_splits=3 and total_samples=10, then base_fold_size=3 and remainder=1, fold_sizes = [4,3,3]
        fold_sizes.append(base_fold_size + (1 if i < remainder else 0))
    
    #initialize the vecotr containing the indices of the folds
    current = 0
    folds = []
    for size in fold_sizes:
        start, stop = current, current + size
        folds.append(indices[start:stop])
        current = stop

    return folds

# Cross-validation using the get_folds function
def cross_val(X, y, n_splits=5, seed=42):
    folds = get_folds(X, y, n_splits=n_splits, seed=seed)
    accuracies = []

    for i in range(n_splits):
        # initialize the test set
        test_idx = folds[i]
        # initialize the training set (union of all the other folds)
        train_idx = np.hstack([folds[j] for j in range(n_splits) if j != i])

        # split the data into training and test sets
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        # Train the tree on the training fold
        tree = fit_decision_tree(X_train, y_train, max_depth=3, min_impurity_improvement=0.0, cat_features_idx=[])

        # Predict on the validation fold
        y_pred = predict(tree, X_test, cat_features_idx=[])

        # Evaluate accuracy
        acc = accuracy(y_test, y_pred)
        accuracies.append(acc)
        print(f"Fold {i+1} accuracy: {acc:.4f}")

    print(f"\nAverage cross-validation accuracy: {np.mean(accuracies):.4f}")
    return accuracies

NOTE: Using cross-validation on a decision tree is different from training a random forest. With cross-validation, we are not building a model with stronger generalization capabilities — we are simply obtaining a more robust estimate of the performance of a single decision tree model.
In contrast, a random forest combines many decision trees to create a new model that typically generalizes better to unseen data by reducing variance through averaging.

# Nested Cross-Validation

We use **nested cross-validation** when the training process involves **hyperparameter tuning**, hence choosing the hyperparameters (e.g., the regularization strength) that yield the best model performance.

In addition to selecting optimal hyperparameters, we also want a **reliable estimate of the model’s generalization performance** (as before). A straightforward strategy might be to split the dataset into **training**, **validation**, and **test** sets: we tune hyperparameters on the validation set and evaluate final performance on the test set. However, this approach is sensitive to how the data is split.

To address this, **nested cross-validation** uses a two-loop structure:

- The **inner loop** performs hyperparameter tuning using cross-validation. For each set of hyperparameters, the model is trained and validated across *K* different splits of the training data.
- The **outer loop** evaluates the model’s generalization performance. For each outer split, the inner loop selects the best hyperparameters on the training portion, and the resulting model is tested on the outer test fold.

This two-tiered structure ensures that:
- **Hyperparameter tuning is done without peeking at the outer test data**, preventing overfitting.
- **Performance evaluation is based on multiple test folds**, giving a more robust estimate of generalization.

After nested cross-validation is complete, we can retrain the model on the **entire dataset**, using the **most frequently selected hyperparameters** (e.g., the mode from the outer loop), to create the final model.

Let's see it on decision trees!

## With scikit-learn

In [ ]:
from sklearn.model_selection import KFold
from itertools import product
import numpy as np

def nested_cross_val_sklearn(X, y, cat_features_idx, 
                     outer_splits=5, inner_splits=3,
                     param_grid={'max_depth': [2, 3, 4], 'min_impurity_improvement': [0.0, 0.01]}):

    outer_cv = KFold(n_splits=outer_splits, shuffle=True, random_state=42)
    outer_scores = []
    best_params_list = []

    # Extract hyperparameter names and values
    param_names = list(param_grid.keys())
    param_values_lists = [param_grid[name] for name in param_names]

    # Generate all combinations manually (supports up to 2 hyperparameters for simplicity)
    param_combinations = []
    for val1 in param_values_lists[0]:
        for val2 in param_values_lists[1]:
            param_combinations.append((val1, val2))

    for train_val_idx, test_idx in outer_cv.split(X): #outer_cv.split(X) returns the indices of the training and test sets
        X_train_val, X_test = X[train_val_idx], X[test_idx]
        y_train_val, y_test = y[train_val_idx], y[test_idx]

        # Inner loop for hyperparameter tuning
        inner_cv = KFold(n_splits=inner_splits, shuffle=True, random_state=42)
        inner_scores = []

        # consider all possible combinations of hyperparameters and train K models using these hyperparameters
        for param_values in param_combinations: # param_values is a tuple of (max_depth, min_impurity_improvement)
            avg_val_score = 0

            for inner_train_idx, inner_val_idx in inner_cv.split(X_train_val): #inner_cv.split(X_train_val) returns the indices of the training and test sets
                X_inner_train, X_inner_val = X_train_val[inner_train_idx], X_train_val[inner_val_idx]
                y_inner_train, y_inner_val = y_train_val[inner_train_idx], y_train_val[inner_val_idx]

                # Train the model with the current hyperparameters
                tree = fit_decision_tree(
                    X_inner_train, y_inner_train,
                    max_depth=param_values[0],
                    min_impurity_improvement=param_values[1],
                    cat_features_idx=cat_features_idx
                )
                # Predict on the validation fold
                y_pred = predict(tree, X_inner_val, cat_features_idx)
                # Evaluate accuracy
                score = accuracy(y_inner_val, y_pred)
                # Accumulate the score (of this combination of hyperparameters)
                avg_val_score += score
            # Average the score over inner folds
            avg_val_score /= inner_splits
            # we save a tuple containing the average score and the hyperparameters
            inner_scores.append((avg_val_score, param_values))

        # Choose best parameters from inner loop
        best_inner_score, best_params = max(inner_scores, key=lambda x: x[0])
        best_params_list.append(best_params)

        # Retrain on the full train+val set with best params
        final_tree = fit_decision_tree(
            X_train_val, y_train_val,
            max_depth=best_params[0],
            min_impurity_improvement=best_params[1],
            cat_features_idx=cat_features_idx
        )
        final_predictions = predict(final_tree, X_test, cat_features_idx)
        test_score = accuracy(y_test, final_predictions)
        outer_scores.append(test_score)

    print(f"Average accuracy across outer folds: {np.mean(outer_scores):.4f}")
    print("Best params for each fold:")
    for i, params in enumerate(best_params_list):
        param_dict = {name: val for name, val in zip(param_names, params)}
        print(f" Fold {i+1}: {param_dict}")

    return np.mean(outer_scores), best_params_list

In [ ]:
train_accuracy, best_params = nested_cross_val_sklearn(
    X, y,
    cat_features_idx=[],
    outer_splits=5,
    inner_splits=3,
    param_grid={'max_depth': [2, 3, 4], 'min_impurity_improvement': [0.0, 0.01]}
)

## Without scikit-learn

In [ ]:
import numpy as np
from itertools import product

def get_folds(X, y, n_splits=5, seed=42):
    np.random.seed(seed)
    indices = np.random.permutation(len(X))
    
    total_samples = len(X)
    base_fold_size = total_samples // n_splits
    remainder = total_samples % n_splits

    fold_sizes = []
    for i in range(n_splits):
        fold_sizes.append(base_fold_size + (1 if i < remainder else 0))

    current = 0
    folds = []
    for size in fold_sizes:
        start, stop = current, current + size
        folds.append(indices[start:stop])
        current = stop
    return folds

def nested_cross_val(X, y, cat_features_idx, 
                     outer_splits=5, inner_splits=3,
                     param_grid={'max_depth': [2, 3, 4], 'min_impurity_improvement': [0.0, 0.01]}):

    outer_folds = get_folds(X, y, n_splits=outer_splits, seed=42)
    outer_scores = []
    best_params_list = []

    # Generate all combinations of hyperparameters
    param_combinations = list(product(*param_grid.values()))
    param_names = list(param_grid.keys())

    for i, test_idx in enumerate(outer_folds):
        train_val_idx = np.hstack([fold for j, fold in enumerate(outer_folds) if j != i])
        X_train_val, X_test = X[train_val_idx], X[test_idx]
        y_train_val, y_test = y[train_val_idx], y[test_idx]

        # Inner loop for hyperparameter tuning
        inner_folds = get_folds(X_train_val, y_train_val, n_splits=inner_splits, seed=42)
        inner_scores = []

        for param_values in param_combinations:
            avg_val_score = 0

            for k, val_idx in enumerate(inner_folds):
                train_idx = np.hstack([fold for j, fold in enumerate(inner_folds) if j != k])
                X_inner_train, X_inner_val = X_train_val[train_idx], X_train_val[val_idx]
                y_inner_train, y_inner_val = y_train_val[train_idx], y_train_val[val_idx]

                tree = fit_decision_tree(
                    X_inner_train, y_inner_train,
                    max_depth=param_values[0],
                    min_impurity_improvement=param_values[1],
                    cat_features_idx=cat_features_idx
                )
                y_pred = predict(tree, X_inner_val, cat_features_idx)
                score = accuracy(y_inner_val, y_pred)
                avg_val_score += score

            avg_val_score /= inner_splits
            inner_scores.append((avg_val_score, param_values))

        # Choose best parameters from inner loop
        # max returns the tuple with the highest score (key=lambda x: x[0] means "use the first element of the tuple as the key for comparison")
        best_inner_score, best_params = max(inner_scores, key=lambda x: x[0]) 
        best_params_list.append(best_params)

        # Retrain on the full train+val set with best params
        final_tree = fit_decision_tree(
            X_train_val, y_train_val,
            max_depth=best_params[0],
            min_impurity_improvement=best_params[1],
            cat_features_idx=cat_features_idx
        )
        # Predict on the test fold
        final_predictions = predict(final_tree, X_test, cat_features_idx)
        test_score = accuracy(y_test, final_predictions)
        # Evaluate accuracy
        outer_scores.append(test_score)

    # Report average performance
    print(f"Average accuracy across outer folds: {np.mean(outer_scores):.4f}")
    print("Best params for each fold:")
    for i, params in enumerate(best_params_list):
        param_dict = {name: val for name, val in zip(param_names, params)}
        print(f" Fold {i+1}: {param_dict}")

    return np.mean(outer_scores), best_params_list


In [ ]:
train_accuracy, best_params = nested_cross_val(
    X, y,
    cat_features_idx=[],
    outer_splits=5,
    inner_splits=3,
    param_grid={'max_depth': [2, 3, 4], 'min_impurity_improvement': [0.0, 0.01]}
)

# Random Forest
Decision trees offer advantages in terms of interpretability and computational efficiency. However, relying on a **single tree** for classification can lead to **high variance** and overfitting.

To address this, we build an **ensemble** of multiple decision trees, known as a **forest**. Instead of training just one tree, we train several trees on **slightly different versions** of the training dataset. At test time, we predict the class that receives the majority vote across all trees.

What does **slightly different versions** of the training dataset mean?

Each tree in the forest is trained on a **randomly sampled subset** of the training data. This randomness introduces diversity among the trees, reducing variance and improving generalization.

## Bagging (Bootstrap Aggregating)
Bagging involves creating **bootstrap samples** from the original training dataset. This means:

- Given a dataset with $n$ samples, we randomly select $n$ samples **with replacement** to create a new training set.
- Since sampling is done **with replacement**, some samples may appear multiple times, while others may not appear at all.

A useful analogy:  
> Imagine an urn containing all the training samples. You randomly pick a sample, record it in the new dataset, then **return it to the urn** before drawing again. This process continues until you have selected $n$ samples.

Each tree in the forest is trained on a different bootstrap sample, ensuring variability in the dataset used for training.

## Feature Bagging
If certain features are **strong predictors**, they are likely to be selected repeatedly across multiple trees. This can lead to high correlation among the trees, reducing the effectiveness of the ensemble.

To mitigate this, we introduce **feature bagging**:

- At **each split** within each tree, we randomly select a **subset of features** instead of considering all features.
- The number of features selected at each split is typically **$\sqrt{p}$**, where $p$ is the total number of features.

This forces each tree to explore different features, reducing overfitting.

Unlike **Bootstrap Aggregating (Bagging)**, where we only modify the dataset used to train each tree, **feature bagging** alters the way the tree itself is built. Specifically, the sampling of $\sqrt{p}$ features occurs *at every split*, meaning that the best split is chosen only from the sampled features. As a result, we must modify the split function.  

## 🧪 Exercise: Re-Implement Random Forest with Bootstrap Aggregating and Feature Bagging

Re-implement the Random Forest algorithm using the new implementation style (that does **not** use classes). Test it on the Wine dataset.

### 💡 Hints

- The implementation of **bootstrap aggregating** will be very similar to what you did in **Lab 5**.
- The implementation of **feature bagging** will require more edits.


In [ ]:
def bootstrap_sample(X, y):
    # select n samples from X with replacement
    ...
    # filter X and y based on indices
    ...

def fit_bagging(X, y, T, max_depth): 
    """Fit T trees using bagging"""
    # forest
    trees = []
    ...

def predict_bagging(X, trees): 
    # matrix where each row corresponds to a different tree (each column correspond to a different sample)
    predictions = ...
    y_pred = []
    confidences = []
    # iterate through the columns
    for i in range(predictions.shape[1]):
        # for column (sample) i, consider all rows (all predictions of all trees)
        ...
        # get the number of times each class appears in the column 
        # (array containing the number of times class 0 appears, the number of times class 1 appears ...)
        ...
        # get the class that appears most frequently
        ...
        # we compute the confidence interval as the proportion of times that the majority class gets predicted
        ...
        # add the prediction to the prediction list
        ...
        # add the confidence to the confidence list
        ...
    return np.array(y_pred), np.array(confidences)

In [ ]:
from sklearn import datasets
from sklearn.utils import shuffle

wine = datasets.load_wine()

# shuffle the data to make sure that the classes are not ordered in any way
np.random.seed(0)
X, y = shuffle(wine["data"], wine["target"], random_state=0)

# divide in train and test set stratified
X_train, X_test, y_train, y_test = ...

# Train and evaluate
trees = ...
y_pred, confidences = ...

# Accuracy
acc = ...
print(f"Bagging Accuracy on Wine test set: {acc:.4f}")

In [ ]:
def find_best_split_feature_and_threshold_fb(X, y, cat_features_idx): 
    """Find the best feature and threshold to split the data based on Gini impurity."""
    n_samples, n_features = X.shape

    best_gini = np.inf          # we want to minimize the Gini Impurity
    best_feature = None
    best_threshold = None

    ...
    
    return best_feature, best_threshold

In [ ]:
# Now we have to modify this function in order to use the newer version of find_best_split_feature_and_threshold_fb
def fit_decision_tree_fb(X_train, y_train, max_depth, min_impurity_improvement, cat_features_idx):
    """Fit a decision tree classifier to the training data. 
    We use an iterative approach to build the tree.
    The tree is represented by the Node class, which defines the root.
    Each Node has two attributes: left and right, representing its child nodes.
    These child nodes are also instances of the Node class, recursively forming the tree structure.
    
    Parameters
    ----------
    X_train : np.ndarray
        The training data, shape (n_samples, n_features)
    y_train : np.ndarray
        The target values, shape (n_samples,)
    max_depth : int
        The maximum depth of the tree
    min_impurity_improvement : float
        The minimum impurity improvement required to split a node
    cat_features_idx : list of int
        The indices of the categorical features in X_train. If 
    """
    root = Node(feature=None, threshold=None, prediction=None, left_node=None, right_node=None, is_leaf=False)

    queue = [(0, X_train, y_train, root)] # (depth, X_subset, y_subset, node object)


    while len(queue) > 0: 

        ...
        ...
        ...
        ...
        

        if is_cat: 
            mask_left_child = (X_subset[:, split_feature] == split_threshold)
            X_left_child = X_subset[mask_left_child]
            y_left_child = y_subset[mask_left_child]

            mask_right_child = (X_subset[:, split_feature] != split_threshold)
            X_right_child = X_subset[mask_right_child]
            y_right_child = y_subset[mask_right_child]

        else: # if continous
            mask_left_child = (X_subset[:, split_feature] <= split_threshold)
            X_left_child = X_subset[mask_left_child]
            y_left_child = y_subset[mask_left_child]

            mask_right_child = (X_subset[:, split_feature] > split_threshold)
            X_right_child = X_subset[mask_right_child]
            y_right_child = y_subset[mask_right_child]

        # check if the improvement is too small: if so, we set the node as a leaf
        ...

        # else we can go on with splitting.
        # update the node 
        node.feature = ...
        node.threshold = ...
        node.prediction = ...
        node.left_node = ...
        node.right_node = ...
        node.is_leaf = ...

        # add left and right children to the queue
        if len(X_left_child) > 0 and len(X_right_child) > 0: 
            queue.append((depth + 1, X_left_child, y_left_child, node.left_node))
            queue.append((depth + 1, X_right_child, y_right_child, node.right_node))

    return root

def fit_bagging_fb(X, y, T, max_depth): 
    """Fit T trees using bagging"""
    # forest
    trees = []
    ...

In [ ]:
# Train and evaluate
trees = fit_bagging_fb(X_train, y_train, T=10, max_depth=3)
y_pred, confidences = predict_bagging(X_test, trees)

# Accuracy
acc = accuracy(y_test, y_pred)
print(f"Feature Bagging Accuracy on Wine test set: {acc:.4f}")

# Implement Isolation Forest (alternative version)

- visualization of individual splits: step-by-step implementation
- implementation inside a function, not with a class
- note: dataset bootstrap uses replace=True
- exercise: with isolation forest we do not have labels

Let's visualize what we want to do step by step.

In [ ]:
# First, create a dataset
np.random.seed(0) # for reproducibility

# random gaussian centered at (0,0) with std=1
X_normal = np.random.randn(500, 2)

plt.scatter(X_normal[:, 0], X_normal[:, 1], s=5)
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("Random Gaussian Dataset")
plt.show()

In [ ]:
np.random.seed(1) # for reproducibility

# Now we can add some anomalies, which are points that are far away from the rest of the data
n_anomalies = 10
X_anomalies = np.random.uniform(low=X_normal.max(), high=X_normal.max() + 3, size=(n_anomalies, 2))
X_anomalies *= np.random.choice([-1, 1], size=X_anomalies.shape)

plt.scatter(X_normal[:, 0], X_normal[:, 1], label="Normal Data")
plt.scatter(X_anomalies[:, 0], X_anomalies[:, 1], color="red", label="Anomalies", marker="x")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("Random Gaussian Dataset with Anomalies")
plt.legend()

The isolation tree construction works like this:
1. split the dataset into two parts, with a linear separation boundary (e.g. a line or a plane in higher dimensions)
2. take each of the two subsets (which we call `left` and `right`) and repeat the process

The inference is then:
1. apply the splits to the new data point
2. return the depth of the leaf node where the point ends up

Remember that this procedure is unsupervised, so we do not have labels when we build the tree.

Let's begin by visualizing the construction:

In [ ]:
np.random.seed(0) # for reproducibility

# the whole dataset contains both normal data and anomalies
X = np.vstack((X_normal, X_anomalies))
y = np.hstack((np.zeros(X_normal.shape[0]), np.ones(X_anomalies.shape[0])))

# now shuffle the dataset
indices = np.random.permutation(len(X))
X = X[indices]
y = y[indices]

In [ ]:
np.random.seed(0) # for reproducibility

# first we select a random feature where we will split the data: in this case we have 2 features
# so we can select 0 or 1
feature_idx = np.random.randint(0, X.shape[1])
print("We select the feature:", feature_idx)

In [ ]:
# Now we have to look at the ranges of the features
feature_value_min, feature_value_max = X[:, feature_idx].min(), X[:, feature_idx].max()

plt.scatter(X[:, 0], X[:, 1], s=5)
# draw a vertical line at the feature value
plt.axvline(x=feature_value_min, color="red", linestyle="--", label=f"Feature {feature_idx} Min")
plt.axvline(x=feature_value_max, color="blue", linestyle="--", label=f"Feature {feature_idx} Max")
# draw a quiver on the axis of the feature {feature_idx}
plt.quiver(feature_value_min, 0, feature_value_max - feature_value_min, 0, angles="xy", scale_units="xy", scale=1, color="black", label=f"Feature {feature_idx} axis")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("Feature Ranges")
plt.legend()

Now, inside that range, we select a random value on that feature axis, which will be the split point, identifying the line (or plane in higher dimensions) that will split the dataset into two parts.

In [ ]:
np.random.seed(0) # for reproducibility

feature_split_value = np.random.uniform(low=feature_value_min, high=feature_value_max)
plt.scatter(X[:, 0], X[:, 1], s=5)
# draw a vertical line at the feature value
plt.axvline(x=feature_split_value, color="green", linestyle="--", label=f"Feature {feature_idx} Split")
plt.legend()

In [ ]:
# Now that we have a random line to split the data, we can use it to split the dataset into 2 parts
# which we will call left and right

# the left part of the dataset is the one where the feature value is less than or equal to the split value
X_left = X[X[:, feature_idx] <= feature_split_value]
# the right part of the dataset is the one where the feature value is greater than the split value
X_right = X[X[:, feature_idx] > feature_split_value]

plt.scatter(X_left[:, 0], X_left[:, 1], color="blue", label="Left Part")
plt.scatter(X_right[:, 0], X_right[:, 1], color="red", label="Right Part")
plt.axvline(x=feature_split_value, color="green", linestyle="--", label=f"Feature {feature_idx} Split")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("Split Dataset")
plt.legend()

Now we repeat that procedure in the same way for the two parts, `X_left` and `X_right` by treating them as if they were the dataset to begin with.

Basically, `X_left` becomes the new `X` for the splitting function and the same goes for `X_right`.

Clearly it is handy to have a function that does this splitting, and in this case we will create the tree in an iterative way, so we will not use recursion, just to see a different approach than the one we used in LAB6.

Since we don't usually grow the tree to the maximum depth, we need to add a correcting factor for the depth, which we call `c(n)`.

In [ ]:
def c(n): 
    """ 
    Average depth of a tree with n_samples samples or equivalently
    average path length of an unsuccessful search in a BST.
    """
    if n <= 1:
        return 0 
    elif n == 2:
        return 1
    else:
        return 2 * (np.log(n-1) + np.euler_gamma - (n-1)/n)

We can call the node of the Isolation tree an `iNode` (for isolation node) to differentiate it from the `Node` of the decision tree.

In [ ]:
@dataclass
class iNode: 
    feature: int | None         # Feature index for split, None for leaf
    threshold: float | None     # Threshold for split, None for leaf
    depth: int | float          # Depth of the node in the tree, float if corrected
    left_node: Optional["iNode"] = None  # Left child node
    right_node: Optional["iNode"] = None # Right child node
    is_leaf: bool = False       # True if leaf node

In [ ]:
np.random.seed(0)  # for reproducibility


def build_tree(X_train, max_depth):
    """Build a decision tree using an iterative approach."""
    root = iNode(
        feature=None,
        threshold=None,
        depth=0,
        left_node=None,
        right_node=None,
        is_leaf=False,
    )
    # compute the average depth of a tree with n_samples nodes
    max_samples = X_train.shape[0]
    c_n = c(max_samples)  # for the correction of the depth

    queue = [(0, X_train, root)]  # (depth, X_subset, parent_node)

    while len(queue) > 0:
        depth, X_subset, node = queue.pop(0)

        # -- leaf conditions --
        # 1. max depth reached
        # 2. only one sample left
        # 3. all samples are the same
        X_all_same = np.all(X_subset == X_subset[0, :], axis=0).all()
        is_leaf = depth >= max_depth or X_subset.shape[0] <= 1 or X_all_same
        if is_leaf:
            node.is_leaf = True
            # we have to correct the depth of the node if we have more than 1 sample left in the node
            node.depth = (node.depth + c(X_subset.shape[0])) / c_n
            continue

        # -- split --
        # this is not a leaf, we need to split the data
        # find a random feature to split the data
        feature_idx = np.random.choice(X_subset.shape[1], size=1)[0]
        # find a random threshold to split the data
        feature_value_min, feature_value_max = (
            X_subset[:, feature_idx].min(),
            X_subset[:, feature_idx].max(),
        )
        feature_split_value = np.random.uniform(
            low=feature_value_min, high=feature_value_max
        )
        # split the data
        X_left = X_subset[X_subset[:, feature_idx] <= feature_split_value]
        X_right = X_subset[X_subset[:, feature_idx] > feature_split_value]

        # save the splitting information in the node
        node.feature = feature_idx
        node.threshold = feature_split_value

        # create the left and right nodes
        node.left_node = iNode(
            feature=None,
            threshold=None,
            depth=depth + 1,
            left_node=None,
            right_node=None,
            is_leaf=False,
        )
        node.right_node = iNode(
            feature=None,
            threshold=None,
            depth=depth + 1,
            left_node=None,
            right_node=None,
            is_leaf=False,
        )
        # add the left and right nodes to the queue
        queue.append((depth + 1, X_left, node.left_node))
        queue.append((depth + 1, X_right, node.right_node))
    return root

In [ ]:
np.random.seed(2) # for reproducibility

# Now we can build the tree
tree = build_tree(X, max_depth=3)
tree

In [ ]:
from utils import plot_tree_with_bboxes, BBox

fig, ax = plt.subplots(figsize=(10, 6))
ax.set_title("Decision Tree with Bounding Boxes")
ax.set_xlabel("Feature 1")
ax.set_ylabel("Feature 2")
ax.scatter(X[:, 0], X[:, 1])
plot_tree_with_bboxes(tree, BBox(X[:, 0].min(), X[:, 0].max(), X[:, 1].min(), X[:, 1].max()), ax)

And now we can define the function that will return the depth of the leaf where a point ends up.

In [ ]:
np.random.seed(0)  # for reproducibility

def get_depth(node, x):
    """
    Given a node and a point x, return the depth of the leaf where x ends up.
    """
    while not node.is_leaf:
        if x[node.feature] ...
            node = ...
        else:
            node = ...
    return node.depth

# we can construct a new tree and compute the depth of each sample
tree = build_tree(X, max_depth=6)

# now plot the depths of the samples
depths = np.array([get_depth(tree, x) for x in X])
plt.scatter(X[:, 0], X[:, 1], c=depths)
plt.colorbar(label="Corrected Depth")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("Corrected Depth of Samples in Decision Tree")
plt.show()

In any case, a single isolation tree may not be a good enough estimator, so we will need to build a forest of isolation trees, named Isolation Forest by just creating an ensemble of isolation trees.

In [ ]:
def train_isolation_forest(X, n_trees, max_depth, max_samples):
    trees = []
    max_samples = min(max_samples, X.shape[0])
    for _ in trange(n_trees, desc="Training Trees"):
        # create a random sample of the data
        X_sampled_indices = ...
        # build a tree on the sample
        ...
    return trees

def predict_isolation_forest(trees, X):
    """
    Predict the anomaly score for each sample in X using the isolation forest.
    The anomaly score is the average depth of the leaf where the sample ends up in all trees.
    """
    depths = np.zeros((X.shape[0], len(trees)))
    for ...

    # compute the anomaly score with the formula
    scores = 2 ** (-avg_depths) # we already divided by c_n while fitting
            
    return scores

isolation_forest = train_isolation_forest(X, n_trees=100, max_depth=6, max_samples=256)
scores = predict_isolation_forest(isolation_forest, X)

plt.scatter(X[:, 0], X[:, 1], c=scores)
plt.colorbar(label="Anomaly Score")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("Anomaly Scores from Isolation Forest")
plt.show()